# Proyecto: Forecasting de Ventas - Rossmann
## Nivel Experto - Pipeline MLOps con Drift y Retraining

---

| | |
|---|---|
| **Universidad** | Universidad Nacional de Ingenieria |
| **Facultad** | Economia, Estadistica y Ciencias Sociales |
| **Programa** | Maestria en Data Science |
| **Curso** | Python for Data Science |
| **Docente** | Melba Torres |
| **Integrante** | [Tu nombre aqui] |
| **Fecha** | 2026-I |

---

## Descripcion del Problema

Este notebook lleva el modelo de forecasting de Rossmann a un **pipeline de produccion** con: persistencia versionada (joblib + metadata), tracking (MLflow/JSON), **deteccion de drift** con PSI + KS test, y **trigger de retraining** automatico cuando el RMSPE supera un umbral o el drift se vuelve severo. Tambien incluye una funcion `predict_forecast` que simula un endpoint REST de forecasting.


---
# FASE 1: Pipeline y Entrenamiento


## 1.1 Importacion de librerias


In [ ]:
import json
import time
import hashlib
import warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['savefig.bbox'] = 'tight'

try:
    import lightgbm as lgb
    LGB_OK = True
except ImportError:
    LGB_OK = False

try:
    import mlflow
    MLFLOW_OK = True
except ImportError:
    MLFLOW_OK = False

RANDOM_STATE = 42
DRIFT_THRESHOLD_PSI = 0.20
RMSPE_TRIGGER_RETRAIN = 0.25
print(f'LightGBM: {LGB_OK}  MLflow: {MLFLOW_OK}')

## 1.2 Configuracion de rutas


In [ ]:
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent
DATA_DIR = PROJECT_ROOT / '01-data'
OUTPUT_DIR = PROJECT_ROOT / '03-resultados' / 'resultados_experto'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1.3 Carga y feature engineering


In [ ]:
def find_csv(directory, keyword):
    for f in directory.glob('*.csv'):
        if keyword.lower() in f.name.lower():
            return f
    raise FileNotFoundError

train = pd.read_csv(find_csv(DATA_DIR, 'train'), low_memory=False, parse_dates=['Date'])
store = pd.read_csv(find_csv(DATA_DIR, 'store'), low_memory=False)
df = train.merge(store, on='Store', how='left')
df = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfYear'] = df['Date'].dt.dayofyear
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)
df['PromoInterval'] = df['PromoInterval'].fillna('None')

encoders = {}
for col in ['StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df['Sales_lag7'] = df.groupby('Store')['Sales'].shift(7)
df['Sales_lag28'] = df.groupby('Store')['Sales'].shift(28)
df['Sales_rmean7'] = df.groupby('Store')['Sales'].shift(7).rolling(7, min_periods=1).mean().reset_index(0, drop=True)
df = df.dropna(subset=['Sales_lag7', 'Sales_lag28']).reset_index(drop=True)

features = [c for c in [
    'Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'StateHoliday',
    'Year', 'Month', 'Day', 'WeekOfYear', 'DayOfYear', 'IsWeekend',
    'StoreType', 'Assortment', 'CompetitionDistance',
    'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
    'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval',
    'Sales_lag7', 'Sales_lag28', 'Sales_rmean7'
] if c in df.columns]
print(f'Filas: {len(df):,}  Features: {len(features)}')

## 1.4 Split temporal y entrenamiento LightGBM


In [ ]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)))

cutoff = df['Date'].max() - pd.Timedelta(days=42)
tr = df[df['Date'] <= cutoff]
te = df[df['Date'] > cutoff]
X_tr, y_tr = tr[features], tr['Sales']
X_te, y_te = te[features], te['Sales']

if LGB_OK:
    params = {'objective': 'regression', 'metric': 'rmse',
              'num_leaves': 64, 'learning_rate': 0.05,
              'feature_fraction': 0.9, 'bagging_fraction': 0.85,
              'bagging_freq': 5, 'min_data_in_leaf': 50,
              'verbose': -1, 'seed': RANDOM_STATE}
    dtr = lgb.Dataset(X_tr, y_tr)
    dte = lgb.Dataset(X_te, y_te, reference=dtr)
    model = lgb.train(params, dtr, num_boost_round=600, valid_sets=[dte],
                       callbacks=[lgb.early_stopping(40), lgb.log_evaluation(0)])
    nombre = 'LightGBM'
else:
    model = GradientBoostingRegressor(n_estimators=200, max_depth=6,
                                       learning_rate=0.05, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr)
    nombre = 'GBR_sklearn'
    params = {'n_estimators': 200, 'max_depth': 6}

pred = model.predict(X_te)
metrics = {
    'model': nombre,
    'rmse': float(np.sqrt(mean_squared_error(y_te, pred))),
    'mae': float(mean_absolute_error(y_te, pred)),
    'r2': float(r2_score(y_te, pred)),
    'rmspe': rmspe(y_te, pred),
}
for k, v in metrics.items():
    print(f'  {k}: {v}')

---
# FASE 2: Persistencia, Tracking y API


## 2.1 Persistencia versionada


In [ ]:
MODEL_VERSION = '1.0.0'
model_path = OUTPUT_DIR / f'modelo_v{MODEL_VERSION}.joblib'
joblib.dump(model, model_path)
joblib.dump(encoders, OUTPUT_DIR / 'encoders.joblib')

meta = {
    'version': MODEL_VERSION,
    'trained_at': datetime.now().isoformat(),
    'model_type': nombre,
    'features': features,
    'params': params,
    'metrics': metrics,
}
(OUTPUT_DIR / 'pipeline_metadata.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(f'Modelo guardado: {model_path}')

## 2.2 Tracking (MLflow o JSON)


In [ ]:
tracking = {'timestamp': datetime.now().isoformat(),
            'params': params, 'metrics': metrics}
if MLFLOW_OK:
    mlflow.set_tracking_uri(f'file://{(OUTPUT_DIR / "mlflow_runs").resolve()}')
    mlflow.set_experiment('rossmann_forecasting')
    with mlflow.start_run():
        for k, v in params.items():
            mlflow.log_param(k, v)
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                mlflow.log_metric(k, float(v))
    print('Tracking via MLflow')
else:
    print('Tracking via JSON local')
(OUTPUT_DIR / 'tracking_log.json').write_text(
    json.dumps(tracking, indent=2, default=str), encoding='utf-8')

## 2.3 API de forecasting (simulada)

`predict_forecast` recibe un dict tipo JSON y devuelve la prediccion + latencia. Es el equivalente a un endpoint FastAPI `/forecast`.


In [ ]:
def predict_forecast(record, model_path, meta_path):
    t0 = time.perf_counter()
    model_loaded = joblib.load(model_path)
    with open(meta_path) as f:
        meta_loaded = json.load(f)
    X_in = pd.DataFrame([record])[meta_loaded['features']]
    pred_val = float(model_loaded.predict(X_in)[0])
    return {
        'forecast': round(pred_val, 2),
        'version': meta_loaded['version'],
        'latency_ms': round((time.perf_counter() - t0) * 1000, 2),
    }

# Demo con 3 registros
muestra = X_te.sample(3, random_state=RANDOM_STATE)
resultados_api = []
for _, row in muestra.iterrows():
    r = predict_forecast(row.to_dict(), model_path, OUTPUT_DIR / 'pipeline_metadata.json')
    r['store'] = int(row['Store']) if 'Store' in row else None
    resultados_api.append(r)
    print(f"  Store={r['store']}  forecast={r['forecast']:.0f}  ({r['latency_ms']} ms)")
pd.DataFrame(resultados_api).to_csv(OUTPUT_DIR / 'demo_api.csv', index=False)

---
# FASE 3: Monitoreo de Drift y Retraining


## 3.1 Deteccion de drift (PSI + Kolmogorov-Smirnov)

Calculamos **PSI** y **KS-test** entre train y test para cada feature. Las features con PSI > 0.2 disparan alerta de drift.


In [ ]:
def psi(expected, actual, buckets=10):
    expected = np.asarray(expected, dtype=float)
    actual = np.asarray(actual, dtype=float)
    breakpoints = np.unique(np.quantile(expected, np.linspace(0, 1, buckets + 1)))
    if len(breakpoints) < 3:
        return 0.0
    e_counts, _ = np.histogram(expected, bins=breakpoints)
    a_counts, _ = np.histogram(actual, bins=breakpoints)
    e_perc = np.clip(e_counts / e_counts.sum(), 1e-6, None)
    a_perc = np.clip(a_counts / max(a_counts.sum(), 1), 1e-6, None)
    return float(np.sum((a_perc - e_perc) * np.log(a_perc / e_perc)))

rows = []
for f in features:
    p = psi(X_tr[f].values, X_te[f].values)
    ks_stat, ks_p = stats.ks_2samp(X_tr[f].values, X_te[f].values)
    flag = 'DRIFT' if p > DRIFT_THRESHOLD_PSI else 'OK'
    rows.append({'feature': f, 'psi': p, 'ks_stat': float(ks_stat),
                 'ks_pvalue': float(ks_p), 'flag': flag})
df_drift = pd.DataFrame(rows).sort_values('psi', ascending=False)
df_drift.to_csv(OUTPUT_DIR / 'drift_report.csv', index=False)
print('Top 10 drift:')
print(df_drift.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(x='psi', y='feature', data=df_drift.head(15),
            ax=ax, palette='RdYlGn_r')
ax.axvline(DRIFT_THRESHOLD_PSI, color='red', linestyle='--',
           label=f'umbral PSI={DRIFT_THRESHOLD_PSI}')
ax.set_title('PSI por feature (train vs test)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_drift_psi.png')
plt.show()

## 3.2 Trigger de retraining


In [ ]:
drift_severo = (df_drift['psi'] > DRIFT_THRESHOLD_PSI).sum()
rmspe_alto = metrics['rmspe'] > RMSPE_TRIGGER_RETRAIN
retrain_needed = (drift_severo > 0) or rmspe_alto

print(f'Features con drift severo: {drift_severo}')
print(f'RMSPE actual ({metrics["rmspe"]:.3f}) > umbral ({RMSPE_TRIGGER_RETRAIN}): {rmspe_alto}')
print(f'\nRetrain necesario: {retrain_needed}')

if retrain_needed:
    print('Reentrenando...')
    if LGB_OK:
        dtr2 = lgb.Dataset(pd.concat([X_tr, X_te]), pd.concat([y_tr, y_te]))
        m2 = lgb.train(params, dtr2, num_boost_round=600,
                        callbacks=[lgb.log_evaluation(0)])
    else:
        m2 = GradientBoostingRegressor(n_estimators=200, max_depth=6,
                                        learning_rate=0.05, random_state=RANDOM_STATE)
        m2.fit(pd.concat([X_tr, X_te]), pd.concat([y_tr, y_te]))
    new_version = '2.0.0'
    joblib.dump(m2, OUTPUT_DIR / f'modelo_v{new_version}.joblib')
    meta['version'] = new_version
    meta['retrained_at'] = datetime.now().isoformat()
    meta['retrain_reason'] = f'drift={drift_severo} features, rmspe={metrics["rmspe"]:.3f}'
    (OUTPUT_DIR / f'metadata_v{new_version}.json').write_text(
        json.dumps(meta, indent=2), encoding='utf-8')
    print(f'Nuevo modelo: modelo_v{new_version}.joblib')

print('\nPipeline experto completado.')

---
# Conclusiones Finales

## Resumen del pipeline

### Fase 1 - Entrenamiento
- LightGBM con features temporales (lags 7/28, rolling mean 7).
- Split temporal con cutoff de 42 dias.

### Fase 2 - Persistencia y API
- Modelo + encoders + metadata persistidos con versionado semantico.
- Tracking en MLflow (o JSON local).
- `predict_forecast` con latencia sub-100ms.

### Fase 3 - Drift y retraining
- PSI + KS detectan drift por feature.
- Trigger automatico de retraining si hay drift severo o RMSPE alto.

## De aqui a produccion

- Envolver `predict_forecast` en FastAPI + autenticacion.
- Orquestar drift check con Airflow (diario).
- Retraining automatico con tests + despliegue via CI/CD.
